# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library, with entity references by `@id` for all components. You will learn how to load metadata, extract record sets, fields, and columns, and perform simple EDA on the FAIR² dataset.

### Dataset Source
The dataset source is provided as a [Croissant schema JSON-LD file](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
We'll now load the schema metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant metadata schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata object (Properties: name, description, identifier, etc)
print(f"Dataset loaded.\nName: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
We will review all available record sets in the dataset, as well as their fields, each referenced by its `@id`. This helps in referencing the correct components when extracting and analyzing data.

In [ ]:
# List all record sets with their @id and field @ids
print("Available record sets and their fields:")

for record_set in dataset.metadata.record_set:
    print(f"  RecordSet: {record_set.id}")
    if hasattr(record_set, 'field'):
        for field in record_set.field:
            print(f"    Field: {field.id}")
            if hasattr(field, 'column'):
                for column in field.column:
                    print(f"      Column: {column.id}")

## 3. Data Extraction
Now, we'll select all main record sets by `@id`, load records from each, and convert to pandas DataFrames for further exploration. Refer to the previous cell for the `@id` to use for record sets.

In [ ]:
# Collect @id for all record sets
record_set_ids = [rs.id for rs in dataset.metadata.record_set]

# For this dataset, there's typically only one main record set containing the core tabular data.
# We'll demonstrate loading all record sets.
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Fields:\n{list(df.columns)}")
        display(df.head(3))
    else:
        print("No records found for this record set.")

# If the dataset contains only one main record set, select it for downstream EDA
main_rs = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Let's process the data: filter records, normalize a numeric field, and group by a key attribute. Replace the field `@id`s as appropriate for your dataset.

Suppose from the overview, we identify the column `cr:age_at_diagnosis` as a numeric field, and `cr:sex` as a grouping field. We'll use these as examples for filtering, normalization, and aggregation.

In [ ]:
# Specify the record set and fields by their @id
record_set_id = main_rs

# Example numeric and group field @ids (replace with those present in your data):
numeric_field_id = 'cr:age_at_diagnosis'  # Example @id for age field
group_field_id = 'cr:sex'                # Example @id for sex

df = dataframes.get(record_set_id)
if df is not None and numeric_field_id in df.columns:
    threshold = 50
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in '{record_set_id}' where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f'{numeric_field_id}_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Field '{numeric_field_id}' or DataFrame not found. Please adjust the field @id as needed.")

## 5. Visualization

Let's visualize the distribution of the numeric field and compare across groups. We'll use matplotlib or seaborn for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

- We have loaded the FAIR² clinical oncology dataset using the Croissant schema and `mlcroissant`.
- Record sets, fields, and columns were referenced and processed using their `@id`s, ensuring reproducibility.
- The data was filtered and normalized for exploratory analysis, and visualizations provided insights into field distributions.
- Update the field `@id`s and EDA as needed for your analysis tasks.

<small>Notebook written for the FAIR² dataset exploration. Refer to [mlcommons/mlcroissant documentation](https://github.com/mlcommons/croissant) for advanced usage and schema features.</small>